# BF520 DMS Functional Score Pipeline

Runs each pipeline step by importing its module. Each step saves its own output files.

| Step | Module | Input | Output folder/ Output csv file |
|------|--------|-------|--------|
| 1 | `CleanPairer` | `variant_counts/` | `results/functional_selections_clean.csv` |
| 2 | `VariantMerger` | `functional_selections_clean.csv` + `variant_counts/` | `results/merged_output/*_merged.csv` |
| 3 | `MutationMapper` | `merged_output/` + `site_numbering_map.csv` | `results/mapped_output/*_merged_mapped.csv` |
| 4 | `FunctionalScoreCalculator` | `mapped_output/` | `results/func_scores_output/*_merged_mapped_func_score.csv` |

## Configuration
All paths are loaded from `config.yaml` — edit there to change any path.

In [ ]:
from config import config

cfg = config.pipeline

VARIANT_COUNTS_DIR    = config.variant_counts_dir
SITE_NUMBERING_MAP    = config.site_numbering_map
FUNCTIONAL_SELECTIONS = config.functional_selections
MERGED_OUTPUT_DIR     = config.merged_output_dir
MAPPED_OUTPUT_DIR     = config.mapped_output_dir
FUNC_SCORES_DIR       = config.func_scores_dir

### Step 1: Pair pre-selection and no-antibody control samples

Scans the `variant_counts/` directory and identifies matching pre-selection (`VSVG_control`) and post-selection (`no-antibody_control`) raw CSV files based on their shared date, rescue batch, and replicate identifiers. These pairs are used to compare mutation frequencies before and after viral growth in the absence of antibody pressure.

**Output:** `results/functional_selections_clean.csv`

In [ ]:
from CleanPairer import CleanPairer

pairer = CleanPairer(
    variant_counts_dir=VARIANT_COUNTS_DIR,
    output_csv=FUNCTIONAL_SELECTIONS,
)
selections = pairer.run()
selections


### Step 2: Merge pre-selection and post-selection variant count files

Reads the selection pairs defined in functional_selections_clean.csv and combines the corresponding pre-selection (VSVG_control) and post-selection (no-antibody_control) variant count datasets. This creates a unified dataset for each selection pair, enabling direct comparison of mutation frequencies before and after selection.

Output: `results/merged_output/*_merged.csv`




In [ ]:
from variantMerger import VariantMerger

merger = VariantMerger(
    variant_count_dir=VARIANT_COUNTS_DIR,
    selection_file=FUNCTIONAL_SELECTIONS,
    output_dir=MERGED_OUTPUT_DIR,
)
merger.run_all()

### Step 3: Standardize BF520 mutation positions

Maps BF520 mutation positions to the standard HIV reference numbering system using a site numbering map. This allows mutations identified in the BF520 background to be compared consistently with published HIV datasets and reference annotations.

**Output:** `results/mapped_output/*_merged_mapped.csv` (one file per selection)

In [ ]:
from MutationMapper import MutationMapper

mapper = MutationMapper(
    input_folder=MERGED_OUTPUT_DIR,
    map_file=SITE_NUMBERING_MAP,
    output_dir=MAPPED_OUTPUT_DIR,
)
mapper.run_all()

### Step 4: Calculate mutation functional scores and functional score variance

Computes a functional score and functional score variance for each mutation by comparing its frequency before selection (VSVG_control) and after no-antibody selection (no-antibody_control)

Output: `results/func_scores_output/*_merged_mapped_func_score.csv` (one file per selection)

In [ ]:
from FunctionalCalculator import FunctionalScoreCalculator

calc = FunctionalScoreCalculator(
    pseudocount=cfg["pseudocount"],
    min_preselection_counts=cfg["min_preselection_counts"],
    min_preselection_frac=cfg["min_preselection_frac"],
)
calc.run_all(
    input_dir=MAPPED_OUTPUT_DIR,
    output_dir=FUNC_SCORES_DIR,
    selections_file=FUNCTIONAL_SELECTIONS,
)
